In [9]:
import pandas as pd
import numpy as np
import json

# 定义文件名
file_signal = r'D:\python_demo\coding\data\原始数据.xlsx'
file_stations = r'D:\python_demo\coding\data\基站经纬度数据.xlsx'
file_mode = r'D:\python_demo\coding\data\出行方式静态数据.xlsx'
file_purpose = r'D:\python_demo\coding\data\出行目的静态数据.xlsx'

output_path = r'D:\python_demo\coding\out.json'

results = {}

try:
    # 优先尝试带表头读取；如果文件没有表头再用 header=None 后推断
    try:
        df_signal = pd.read_excel(file_signal, header=0)
        used_header = True
    except Exception:
        df_signal = pd.read_excel(file_signal, header=None)
        used_header = False

    # 如果已经有名为 'EventTime' 的列，直接使用
    if 'EventTime' not in df_signal.columns:
        # 如果读取时没有表头，尝试把第一行作为列名（常见情况）
        if not used_header and df_signal.shape[0] > 0:
            first_row_vals = df_signal.iloc[0].astype(str).str.strip().tolist()
            # 判断第一行是否看起来像列名（包含 'EventTime' 或包含 'time' 关键词）
            if any('eventtime' == v.lower() or 'event' in v.lower() and 'time' in v.lower() for v in first_row_vals):
                df_signal.columns = first_row_vals
                df_signal = df_signal.iloc[1:].reset_index(drop=True)
            else:
                # 尝试在第一行中模糊匹配包含 'event' 或 'time' 的列名并将其设为 'EventTime'
                found = False
                for i, v in enumerate(first_row_vals):
                    if isinstance(v, str) and ('event' in v.lower() or 'time' in v.lower()):
                        df_signal.columns = first_row_vals
                        df_signal = df_signal.iloc[1:].reset_index(drop=True)
                        df_signal = df_signal.rename(columns={v: 'EventTime'})
                        found = True
                        break
                if not found:
                    # 再尝试在列名（如果存在字符串列名）中模糊查找，或在数据列中寻找可能的时间戳（大数）列
                    candidates = [c for c in df_signal.columns if isinstance(c, str) and ('event' in c.lower() or 'time' in c.lower())]
                    if candidates:
                        df_signal = df_signal.rename(columns={candidates[0]: 'EventTime'})
                    else:
                        # 尝试通过数据内容识别：选择数值列且中位数很大（可能为 ms 时间戳）
                        candidate_col = None
                        for c in df_signal.columns:
                            ser = pd.to_numeric(df_signal[c], errors='coerce')
                            if ser.dropna().shape[0] > 0:
                                med = ser.dropna().median()
                                if med is not None and med > 1e9:
                                    candidate_col = c
                                    break
                        if candidate_col is not None:
                            df_signal = df_signal.rename(columns={candidate_col: 'EventTime'})
                        else:
                            raise KeyError('EventTime')
        else:
            # 读取带表头但没有 EventTime，尝试模糊匹配列名
            candidates = [c for c in df_signal.columns if isinstance(c, str) and ('event' in c.lower() or 'time' in c.lower())]
            if candidates:
                df_signal = df_signal.rename(columns={candidates[0]: 'EventTime'})
            else:
                # 如无法找到，尝试使用列位置猜测（例如第0或第1列）并检测值域
                candidate_col = None
                for c in df_signal.columns:
                    ser = pd.to_numeric(df_signal[c], errors='coerce')
                    if ser.dropna().shape[0] > 0:
                        med = ser.dropna().median()
                        if med is not None and med > 1e9:
                            candidate_col = c
                            break
                if candidate_col is not None:
                    df_signal = df_signal.rename(columns={candidate_col: 'EventTime'})
                else:
                    raise KeyError('EventTime')

    # 到这里应该有 'EventTime' 列；如果没有，上面的逻辑会抛出 KeyError
    # 解析时间：支持毫秒时间戳或标准时间字符串
    try:
        timestamps = pd.to_datetime(df_signal['EventTime'], unit='ms', errors='coerce')
        # 若全部为 NaT，则再尝试不带 unit 的解析
        if timestamps.dropna().shape[0] == 0:
            timestamps = pd.to_datetime(df_signal['EventTime'], errors='coerce')
    except Exception:
        timestamps = pd.to_datetime(df_signal['EventTime'], errors='coerce')

    # 若仍然没有有效时间，记录并继续（避免后续报错）
    if timestamps.dropna().shape[0] == 0:
        print("警告：未识别到有效的 EventTime 值，后续时长/开始/结束时间将设为 None 或 0。")
        start_time = None
        end_time = None
        duration = pd.Timedelta(0)
    else:
        start_time = timestamps.min()
        end_time = timestamps.max()
        duration = end_time - start_time

    # 下面为原有统计代码（确保类型可 JSON 序列化）
    total_records = len(df_signal)
    results['total_records'] = int(total_records)
    print(f"总记录数量: {total_records}")

    # 唯一的用户数（以及用户列表）
    # 注意：原数据中用户列索引可能不同，这里使用 index 1（若不存在会尝试其他列）
    if 1 in df_signal.columns:
        users_col = 1
    else:
        # 尝试找到一个看起来像用户 ID 的列（字符串/数字混合）
        users_candidates = [c for c in df_signal.columns if df_signal[c].dtype == object or pd.api.types.is_integer_dtype(df_signal[c]) or pd.api.types.is_float_dtype(df_signal[c])][:1]
        users_col = users_candidates[0] if users_candidates else 1
    unique_users_list = df_signal[users_col].dropna().astype(str).unique().tolist()
    unique_users_count = len(unique_users_list)
    results['unique_users_count'] = unique_users_count
    results['unique_users_sample'] = unique_users_list[:20]  # 存一部分样本以供检查
    print(f"唯一的用户数: {unique_users_count}")

    # 每个用户平均拿到记录数量
    if unique_users_count > 0:
        avg_records_per_user = total_records / unique_users_count
    else:
        avg_records_per_user = 0
    results['avg_records_per_user'] = float(avg_records_per_user)
    print(f"每个用户平均记录数: {avg_records_per_user:.2f}")

    # 唯一基站数量（兼容没有列名或有列名的情况）
    df_stations = pd.read_excel(file_stations, header=0)
    if 'Code' in df_stations.columns:
        unique_base_stations_observed = df_stations['Code'].dropna().astype(str).unique().tolist()
    else:
        unique_base_stations_observed = df_stations.iloc[:,0].dropna().astype(str).unique().tolist()
    # 仅保留前10条用于输出展示，统计计数仍然保留完整长度
    results['unique_base_stations_observed'] = unique_base_stations_observed[:10]
    results['unique_base_stations_count'] = len(unique_base_stations_observed)
    print(f"观测到的唯一基站: {len(unique_base_stations_observed)} 个样本（示例前5个）: {unique_base_stations_observed[:5]}")

    # 开始/结束/持续时长的 JSON 友好写法
    results['start_time'] = str(start_time) if start_time is not None else None
    results['end_time'] = str(end_time) if end_time is not None else None
    results['duration_total_seconds'] = float(duration.total_seconds()) if duration is not None else 0.0
    results['duration_hours'] = results['duration_total_seconds'] / 3600 if results['duration_total_seconds'] else 0.0
    results['duration_days'] = results['duration_total_seconds'] / (3600 * 24) if results['duration_total_seconds'] else 0.0

    # 核心信令数据集重复记录的数量和比例
    duplicate_rows = int(df_signal.duplicated().sum())
    total_records = len(df_signal)
    results['duplicate_records'] = duplicate_rows # 转换为标准int以便JSON序列化
    if total_records > 0:
        duplicate_proportion = duplicate_rows / total_records
    else:
        duplicate_proportion = 0
    results['duplicate_proportion'] = float(duplicate_proportion)
    print(f"重复记录数量: {duplicate_rows} (占比: {duplicate_proportion:.4f})")

    # 后续对静态数据的处理（保持原逻辑，不会触发 EventTime KeyError）
    station_info = {}
    df_mode = pd.read_excel(file_mode, header=0)
    if 'Mot' in df_mode.columns and 'Loc' in df_mode.columns:
        unique_mode_types = df_mode['Mot'].dropna().unique().tolist()
        for mode in unique_mode_types:
            stations_for_mode = df_mode[df_mode['Mot'] == mode]['Loc'].astype(str).str.replace(r'\\(地铁站\\)', '', regex=True).unique().tolist()
            # 若为地铁，仅保留前10条到 station_info
            if mode == '地铁':
                station_info[f'{mode}_number'] = len(stations_for_mode)
                station_info[mode] = stations_for_mode[:10]
            else:
                station_info[f'{mode}_number'] = len(stations_for_mode)
                station_info[mode] = stations_for_mode[:10]
    else:
        unique_mode_types = df_mode.iloc[:,1].dropna().unique().tolist()
        for mode in unique_mode_types:
            stations_for_mode = df_mode[df_mode.iloc[:,1] == mode].iloc[:,2].astype(str).str.replace(r'\\(地铁站\\)', '', regex=True).unique().tolist()
            if mode == '地铁':
                station_info[f'{mode}_number'] = len(stations_for_mode)
                station_info[mode] = stations_for_mode[:10]
            else:
                station_info[f'{mode}_number'] = len(stations_for_mode)
                station_info[mode] = stations_for_mode[:10]

    results['related_info_travel_mode'] = {
        'total_entries': int(len(df_mode)),
        'unique_mode_types': unique_mode_types,
        'station_subways_sample': station_info.get('地铁', [])[:10],
        'station_info': station_info,
    }
    print(f"出行方式数据条目: {len(df_mode)}")

    df_purpose = pd.read_excel(file_purpose, header=0)
    if 2 in df_purpose.columns:
        purpose_unique = df_purpose[2].dropna().unique().tolist()
    else:
        purpose_unique = df_purpose.iloc[:,2].dropna().unique().tolist() if df_purpose.shape[1] > 2 else []

    results['related_info_travel_purpose'] = {
        'total_entries': int(len(df_purpose)),
        'unique_purpose_types': purpose_unique
    }
    print(f"出行目的数据条目: {len(df_purpose)}")

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    print(f"\n成功！所有统计信息已保存到: {output_path}")
    print("\n--- JSON 结果概览 ---")
    print(json.dumps(results, ensure_ascii=False, indent=2))

except FileNotFoundError as e:
    print(f"错误：文件未找到. {e}")
except KeyError as e:
    print(f"处理数据时发生错误: 无法定位列 {e}，请检查输入文件的列名或表头格式。")
except Exception as e:
    print(f"处理数据时发生错误: {e}")

总记录数量: 532828
唯一的用户数: 528352
每个用户平均记录数: 1.01
观测到的唯一基站: 49954 个样本（示例前5个）: ['16481-536531', '16481-536532', '16770-1080845', '16484-1111411', '16459-1121547']
重复记录数量: 2568 (占比: 0.0048)
观测到的唯一基站: 49954 个样本（示例前5个）: ['16481-536531', '16481-536532', '16770-1080845', '16484-1111411', '16459-1121547']
重复记录数量: 2568 (占比: 0.0048)
出行方式数据条目: 968
出行目的数据条目: 1273

成功！所有统计信息已保存到: D:\python_demo\coding\out.json

--- JSON 结果概览 ---
{
  "total_records": 532828,
  "unique_users_count": 528352,
  "unique_users_sample": [
    "1538498482636",
    "1538502547368",
    "1538481509242",
    "1538526105074",
    "1538526153986",
    "1538527924038",
    "1538530014006",
    "1538544930607",
    "1538526826182",
    "1538524523725",
    "1538566688867",
    "1538569125656",
    "1538570726161",
    "1538571972888",
    "1538575863678",
    "1538575888250",
    "1538576395802",
    "1538578553370",
    "1538578877783",
    "1538579072410"
  ],
  "avg_records_per_user": 1.0084716249772878,
  "unique_base_stations_ob

In [2]:
file_signal = r'D:\python_demo\coding\data\原始数据.xlsx'
file_stations = r'D:\python_demo\coding\data\基站经纬度数据.xlsx'
file_mode = r'D:\python_demo\coding\data\出行方式静态数据.xlsx'
file_purpose = r'D:\python_demo\coding\data\出行目的静态数据.xlsx'

output_path = r'D:\python_demo\coding\out.json'
df_signal = pd.read_excel(file_signal, header=None)
df_stations = pd.read_excel(file_stations, header=None)
df_mode = pd.read_excel(file_mode, header=None)
df_purpose = pd.read_excel(file_purpose, header=None)
print(df_signal.head(2), df_stations.head(2), df_mode.head(2), df_purpose.head(2))

               0                   1       2          3               4  \
0      EventTime              UserID  LineNo  StationNo        DeviceID   
1  1538498482636  460000095007329090   16789   67567924  86137666647316   

               5   6         7           8         9  
0     RecordTime NaN  Reserved  StatusCode  RouteTag  
1  1538498481670 NaN                     4   #*#6137               0          1             2
0         Lng        Lat          Code
1  123.919968  41.449322  16481-536531             0          1    2         3   4
0         Lng        Lat  Mot       Loc  Dt
1  123.417133  41.868337   地铁  三台子(地铁站)   2             0          1     2        3
0         Lng        Lat  type     name
1  123.457628  41.722539    居住  万科·明天广场
